# Birkin vs Not-Birkin CNN (NumPy Layers)

This notebook builds a small CNN **using the `layers/` folder modules** and lets you tune hyperparameters. It also appends the best CNN run to `outputs/pipeline_report.txt`.

In [ ]:

from __future__ import annotations

from pathlib import Path
from itertools import product

import numpy as np
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from layers import Conv2D, MaxPool2D, Flatten, Linear, ReLU, Softmax, Sequential, CrossEntropyLoss


## Hyperparameters (tune here)

In [ ]:

# Data parameters
BIRKIN_DIRS = [Path("Data/Birkin"), Path("Data/birkins")]
OTHER_DIR = Path("Data/other")
IMAGE_SIZE = 16
TEST_SIZE = 0.2
SEED = 42
MAX_SAMPLES_PER_CLASS = 120  # reduce this if training is slow

# Search space
LEARNING_RATES = [0.01, 0.005]
BATCH_SIZES = [8, 16]
EPOCHS = [3]
CONV_CHANNELS = [4, 8]

# Output report path
REPORT_PATH = Path("outputs/pipeline_report.txt")


In [ ]:

IMAGE_PATTERNS = ("*.jpg", "*.jpeg", "*.png")

def list_image_files(folder: Path) -> list[Path]:
    files = []
    for pattern in IMAGE_PATTERNS:
        files.extend(sorted(folder.glob(pattern)))
    return files

def load_image(path: Path, size: int) -> np.ndarray:
    image = Image.open(path).convert("L").resize((size, size), Image.Resampling.LANCZOS)
    arr = np.asarray(image, dtype=np.float32) / 255.0
    return arr

def build_dataset(size: int, max_samples_per_class: int | None = None):
    birkin_files = []
    for d in BIRKIN_DIRS:
        birkin_files.extend(list_image_files(d))
    other_files = list_image_files(OTHER_DIR)

    if max_samples_per_class is not None:
        birkin_files = birkin_files[:max_samples_per_class]
        other_files = other_files[:max_samples_per_class]

    x_birkin = [load_image(p, size) for p in birkin_files]
    x_other = [load_image(p, size) for p in other_files]

    x = np.array(x_birkin + x_other, dtype=np.float32)
    y = np.array([1] * len(x_birkin) + [0] * len(x_other), dtype=np.int64)

    x = x[:, None, :, :]  # (N, C, H, W)
    return x, y


In [ ]:

def make_model(image_size: int, conv_channels: int):
    pooled = image_size // 2
    hidden = 32

    model = Sequential([
        Conv2D(in_channels=1, out_channels=conv_channels, kernel_size=3, stride=1, padding=1),
        ReLU(),
        MaxPool2D(kernel_size=2, stride=2),
        Flatten(),
        Linear(conv_channels * pooled * pooled, hidden),
        ReLU(),
        Linear(hidden, 2),
        Softmax(),
    ])
    return model


def iter_batches(x, y, batch_size, rng):
    idx = np.arange(len(x))
    rng.shuffle(idx)
    for start in range(0, len(x), batch_size):
        sl = idx[start:start + batch_size]
        yield x[sl], y[sl]


def sgd_step(model: Sequential, lr: float):
    for module in model._modules:
        if hasattr(module, "parameters") and hasattr(module, "grads"):
            for p, g in zip(module.parameters, module.grads):
                if g is not None:
                    p -= lr * g


In [ ]:

def train_once(x_train, y_train, x_val, y_val, lr, batch_size, epochs, conv_channels, seed=42):
    rng = np.random.default_rng(seed)
    model = make_model(IMAGE_SIZE, conv_channels)
    loss_fn = CrossEntropyLoss()

    history = {"loss": [], "val_acc": []}

    for epoch in range(epochs):
        running_loss = []
        for xb, yb in iter_batches(x_train, y_train, batch_size, rng):
            probs = model.forward(xb)
            loss = loss_fn.forward(probs, yb)
            grad = loss_fn.backward()

            model.zero_grad()
            model.backward(grad)
            sgd_step(model, lr)
            running_loss.append(loss)

        val_pred = model.predict(x_val)
        val_acc = float((val_pred == y_val).mean())
        history["loss"].append(float(np.mean(running_loss)))
        history["val_acc"].append(val_acc)
        print(f"epoch={epoch+1}/{epochs} loss={history['loss'][-1]:.4f} val_acc={val_acc:.4f}")

    return model, history


## Run hyperparameter search

In [ ]:

x, y = build_dataset(size=IMAGE_SIZE, max_samples_per_class=MAX_SAMPLES_PER_CLASS)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

x_train_sub, x_val, y_train_sub, y_val = train_test_split(
    x_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

best = None
search_results = []

for lr, bs, ep, ch in product(LEARNING_RATES, BATCH_SIZES, EPOCHS, CONV_CHANNELS):
    print("\n=== trial ===")
    print({"lr": lr, "batch_size": bs, "epochs": ep, "conv_channels": ch})
    model, history = train_once(x_train_sub, y_train_sub, x_val, y_val, lr, bs, ep, ch, seed=SEED)
    score = history["val_acc"][-1]

    row = {
        "lr": lr,
        "batch_size": bs,
        "epochs": ep,
        "conv_channels": ch,
        "val_acc": score,
        "history": history,
        "model": model,
    }
    search_results.append(row)

    if best is None or score > best["val_acc"]:
        best = row

print("\nBest config:", {k: best[k] for k in ["lr", "batch_size", "epochs", "conv_channels", "val_acc"]})


## Retrain on full training split and evaluate on test split

In [ ]:

best_model, best_history = train_once(
    x_train,
    y_train,
    x_test,
    y_test,
    best["lr"],
    best["batch_size"],
    best["epochs"],
    best["conv_channels"],
    seed=SEED,
)

y_pred = best_model.predict(x_test)
acc = float((y_pred == y_test).mean())
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=["not_birkin", "birkin"], digits=4)

print(f"Test accuracy: {acc:.4f}")
print(cm)
print(report)


## Append CNN result to `outputs/pipeline_report.txt`

In [ ]:

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
with REPORT_PATH.open("a", encoding="utf-8") as f:
    f.write("\n\n================ CNN (NumPy layers) ================\n")
    f.write(f"image_size={IMAGE_SIZE}x{IMAGE_SIZE}\n")
    f.write(f"train_samples={len(x_train)}\n")
    f.write(f"test_samples={len(x_test)}\n")
    f.write(f"max_samples_per_class={MAX_SAMPLES_PER_CLASS}\n")
    f.write("best_hyperparameters=\n")
    f.write(f"  learning_rate={best['lr']}\n")
    f.write(f"  batch_size={best['batch_size']}\n")
    f.write(f"  epochs={best['epochs']}\n")
    f.write(f"  conv_channels={best['conv_channels']}\n")
    f.write(f"test_accuracy={acc:.4f}\n")
    f.write("Confusion matrix [[TN, FP], [FN, TP]]:\n")
    f.write(f"{cm}\n\n")
    f.write(report)

print(f"Appended CNN results to: {REPORT_PATH}")
